# ForecastLLM - Week 6 Day 3: Baselines and Evaluation

This week - build a model that predicts future values from historical data.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Baselines and Evaluation  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 3: Baselines and Evaluation

Today we'll build simple forecasting baselines and evaluate them carefully.

These baseline scores become the benchmark that later models must beat.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from week6.data_loader import load_sample_series


In [2]:
TRAIN_FRACTION = 0.8
SEASONAL_PERIOD = 24

In [3]:
# Load one selected M4 hourly series
USE_SYNTHETIC_FALLBACK = False

try:
    ts_df = load_sample_series()
except Exception as e:
    if not USE_SYNTHETIC_FALLBACK:
        raise
    from week6.data_loader import load_synthetic_series
    print(f"Loader failed ({e}); using explicit synthetic fallback")
    ts_df = load_synthetic_series(periods=240)

if not isinstance(ts_df, pd.DataFrame):
    raise TypeError(f"load_sample_series must return a pandas DataFrame, got {type(ts_df)}")
if not {"timestamp", "value"}.issubset(ts_df.columns):
    raise ValueError("Loaded dataframe must include 'timestamp' and 'value' columns")

ts_df = ts_df[["timestamp", "value"]].copy()
ts_df["value"] = pd.to_numeric(ts_df["value"], errors="coerce")
ts_df["timestamp"] = pd.to_datetime(ts_df["timestamp"], errors="coerce")
ts_df = ts_df.dropna(subset=["timestamp", "value"]).sort_values("timestamp").reset_index(drop=True)

min_required = max(40, SEASONAL_PERIOD + 5)
if len(ts_df) < min_required:
    raise RuntimeError(f"Need at least {min_required} rows, got {len(ts_df)}")

split_idx = int(len(ts_df) * TRAIN_FRACTION)
train_df = ts_df.iloc[:split_idx].copy()
test_df = ts_df.iloc[split_idx:].copy()

print(f"Loaded {len(ts_df):,} observations")
print(f"Train rows: {len(train_df):,}, Test rows: {len(test_df):,}")

Loaded 960 observations
Train rows: 768, Test rows: 192


In [4]:
def naive_forecast(series, split_index):
    # one-step-ahead naive forecast: predict with previous observed value
    return series.shift(1).iloc[split_index:]


In [5]:
naive_pred = naive_forecast(ts_df["value"], split_idx)
naive_pred.head()

768    18.7
769    18.1
770    17.7
771    17.2
772    16.8
Name: value, dtype: float64

In [6]:
# That was a first baseline. Add one that handles hourly seasonality (24h cycle).
def seasonal_naive_forecast(series, split_index, seasonal_period=24):
    # one-step-ahead seasonal naive: use value from one full season earlier
    return series.shift(seasonal_period).iloc[split_index:]


In [7]:
seasonal_pred = seasonal_naive_forecast(ts_df["value"], split_idx, SEASONAL_PERIOD)
seasonal_pred.head()

768    17.6
769    17.2
770    16.8
771    16.4
772    16.1
Name: value, dtype: float64

In [8]:
def mean_absolute_error(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.mean(np.abs(y_true - y_pred))


In [9]:
def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denominator = np.abs(y_true) + np.abs(y_pred)
    denominator = np.where(denominator == 0, 1e-9, denominator)
    return 100 * np.mean(2.0 * np.abs(y_true - y_pred) / denominator)


In [10]:
def evaluate_forecast(y_true, y_pred, model_name):
    frame = pd.DataFrame({"y_true": y_true, "y_pred": y_pred}).dropna()
    return {
        "model": model_name,
        "rows": len(frame),
        "mae": mean_absolute_error(frame["y_true"], frame["y_pred"]),
        "smape": smape(frame["y_true"], frame["y_pred"]),
    }


In [11]:
y_true_test = ts_df["value"].iloc[split_idx:]
naive_results = evaluate_forecast(y_true_test, naive_pred, "Naive")
naive_results

{'model': 'Naive',
 'rows': 192,
 'mae': np.float64(0.7578125),
 'smape': np.float64(3.613184304362057)}

In [12]:
seasonal_results = evaluate_forecast(y_true_test, seasonal_pred, "Seasonal Naive (period=24)")
seasonal_results

{'model': 'Seasonal Naive (period=24)',
 'rows': 192,
 'mae': np.float64(0.1697916666666667),
 'smape': np.float64(0.8260256354631111)}

In [13]:
results_df = pd.DataFrame([naive_results, seasonal_results])
results_df

,model,rows,mae,smape
0,Naive,192,0.757812,3.613184
1,Seasonal Naive (period=24),192,0.169792,0.826026


In [14]:
# Lower MAE and lower sMAPE are better
results_df.sort_values(["mae", "smape"]).reset_index(drop=True)

,model,rows,mae,smape
0,Seasonal Naive (period=24),192,0.169792,0.826026
1,Naive,192,0.757812,3.613184


In [15]:
comparison_df = pd.DataFrame({
    "timestamp": ts_df["timestamp"].iloc[split_idx:],
    "actual": y_true_test.values,
    "naive": naive_pred.values,
    "seasonal_naive": seasonal_pred.values,
})
comparison_df.head()

,timestamp,actual,naive,seasonal_naive
768,2000-02-02 00:00:00,18.1,18.7,17.6
769,2000-02-02 01:00:00,17.7,18.1,17.2
770,2000-02-02 02:00:00,17.2,17.7,16.8
771,2000-02-02 03:00:00,16.8,17.2,16.4
772,2000-02-02 04:00:00,16.4,16.8,16.1


In [16]:
comparison_df.tail()

,timestamp,actual,naive,seasonal_naive
955,2000-02-09 19:00:00,24.7,25.4,24.5
956,2000-02-09 20:00:00,23.2,24.7,23.1
957,2000-02-09 21:00:00,21.6,23.2,21.4
958,2000-02-09 22:00:00,20.5,21.6,20.4
959,2000-02-09 23:00:00,19.8,20.5,19.7


In [17]:
comparison_df["naive_abs_error"] = np.abs(comparison_df["actual"] - comparison_df["naive"])
comparison_df["seasonal_abs_error"] = np.abs(comparison_df["actual"] - comparison_df["seasonal_naive"])
comparison_df[["naive_abs_error", "seasonal_abs_error"]].describe()

,naive_abs_error,seasonal_abs_error
count,192.000000,192.000000
mean,0.757812,0.169792
std,0.495151,0.097207
min,0.000000,0.000000
25%,0.400000,0.100000
50%,0.600000,0.200000
75%,1.200000,0.200000
max,1.700000,0.500000


In [18]:
print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(comparison_df.head(10).to_string(index=False))

Train shape: (768, 2)
Test shape: (192, 2)
          timestamp  actual  naive  seasonal_naive  naive_abs_error  seasonal_abs_error
2000-02-02 00:00:00    18.1   18.7            17.6              0.6                 0.5
2000-02-02 01:00:00    17.7   18.1            17.2              0.4                 0.5
2000-02-02 02:00:00    17.2   17.7            16.8              0.5                 0.4
2000-02-02 03:00:00    16.8   17.2            16.4              0.4                 0.4
2000-02-02 04:00:00    16.4   16.8            16.1              0.4                 0.3
2000-02-02 05:00:00    16.1   16.4            15.7              0.3                 0.4
2000-02-02 06:00:00    16.2   16.1            15.8              0.1                 0.4
2000-02-02 07:00:00    17.5   16.2            17.2              1.3                 0.3
2000-02-02 08:00:00    19.1   17.5            18.8              1.6                 0.3
2000-02-02 09:00:00    20.7   19.1            20.4              1.6          

## Baseline visualization

A chronological split is required in forecasting because future values must never leak into training.

Baseline conventions used here:
- naive baseline = `lag_1`
- daily seasonal naive baseline = `lag_24`
- `lag_7` is retained as a short-memory feature (not a seasonal baseline)

These baselines are intentionally simple. If a more advanced method cannot beat them, it usually is not worth keeping.

In [ ]:
# Plot actual test values against both baselines
plot_df = comparison_df.dropna().copy()

plt.figure(figsize=(14, 6))
plt.plot(plot_df["timestamp"], plot_df["actual"], label="Actual")
plt.plot(plot_df["timestamp"], plot_df["naive"], label="Naive")
plt.plot(plot_df["timestamp"], plot_df["seasonal_naive"], label="Seasonal Naive")
plt.title("Test set: actual vs baseline forecasts")
plt.xlabel("Timestamp")
plt.ylabel("Value")
plt.legend()
plt.show()

In [ ]:
# Compare absolute errors over time
plt.figure(figsize=(14, 5))
plt.plot(plot_df["timestamp"], np.abs(plot_df["actual"] - plot_df["naive"]), label="Naive abs error")
plt.plot(plot_df["timestamp"], np.abs(plot_df["actual"] - plot_df["seasonal_naive"]), label="Seasonal naive abs error")
plt.title("Absolute error by time step")
plt.xlabel("Timestamp")
plt.ylabel("Absolute error")
plt.legend()
plt.show()

In [ ]:
# TODO: compare learned models against baseline forecasts
best_baseline = results_df.sort_values(["mae", "smape"]).iloc[0]["model"]
print(f"Current best baseline: {best_baseline}")

## Next models (deferred)

In the original pricing notebook, this section explored heavier traditional ML models.

For ForecastLLM Day 3, we stop at robust baselines and metrics so Day 4+ models have a clear benchmark.


In [19]:
# TODO: extend to multiple M4 hourly series
multi_series_planned = True
print(f"Multi-series M4 extension planned: {multi_series_planned}")


Multi-series M4 extension planned: True


In [20]:
# Keep a compact artifact for later notebooks
artifacts = {
    "train_rows": len(train_df),
    "test_rows": len(test_df),
    "seasonal_period": SEASONAL_PERIOD,
    "metrics": results_df.to_dict(orient="records"),
}
artifacts

{'train_rows': 768,
 'test_rows': 192,
 'seasonal_period': 24,
 'metrics': [{'model': 'Naive',
   'rows': 192,
   'mae': 0.7578125,
   'smape': 3.613184304362057},
  {'model': 'Seasonal Naive (period=24)',
   'rows': 192,
   'mae': 0.1697916666666667,
   'smape': 0.8260256354631111}]}

In [21]:
# TODO: add rolling-origin evaluation later
rolling_origin_planned = True
print(f"Rolling-origin evaluation planned: {rolling_origin_planned}")

Rolling-origin evaluation planned: True


In [22]:
print("Final baseline table:")
print(results_df.to_string(index=False))

Final baseline table:
                     model  rows      mae    smape
                     Naive   192 0.757812 3.613184
Seasonal Naive (period=24)   192 0.169792 0.826026


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">Baseline forecasting is operationally important: it gives a transparent minimum bar,
            helps detect when advanced models are not adding value, and keeps evaluation grounded in realistic time-ordering.</span>
        </td>
    </tr>
</table>

Later notebooks can compare richer ML and LLM-assisted approaches against these baseline forecasts.